In [5]:
import sys
sys.path.insert(0, '/opt/spark/python')
sys.path.insert(0, '/opt/spark/python/lib/py4j-0.10.9.7-src.zip')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ExploreData") \
    .config("spark.master", "spark://spark-master:7077") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.1


In [9]:
base_path = "hdfs://namenode:8020/data/bronze/home_credit/raw/"

columnsDesc = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "HomeCredit_columns_description.csv")

posCashBalance = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "POS_CASH_balance.csv")

applicationTest = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "application_test.csv")

applicationTrain = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "application_train.csv")

bureauRaw = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "bureau.csv")

bureauBalance = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "bureau_balance.csv")

creditCardBalance = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "credit_card_balance.csv")

installmentsPayments = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "installments_payments.csv")

previousApplication = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "previous_application.csv")

sampleSubmission = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(base_path + "sample_submission.csv")

print("Semua file berhasil dibaca dengan penamaan camelCase!")

Semua file berhasil dibaca dengan penamaan camelCase!


In [31]:
applicationTrain.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullable = true)
 |-- OWN_CAR_AG

In [40]:
applications = spark.sql("""
SELECT 
    SK_ID_CURR AS loanId,
    TARGET AS target,
    NAME_CONTRACT_TYPE AS contractType,
    CODE_GENDER AS gender,
    FLAG_OWN_CAR AS ownCar,
    FLAG_OWN_REALTY AS ownRealty,
    CNT_CHILDREN AS childrenCnt,
    AMT_INCOME_TOTAL AS incomeTotal,
    AMT_CREDIT AS creditAmt,
    AMT_ANNUITY AS annuityAmt,
    AMT_GOODS_PRICE AS goodsPrice,
    NAME_TYPE_SUITE AS suiteType,
    NAME_INCOME_TYPE AS incomeType,
    NAME_EDUCATION_TYPE AS education,
    NAME_FAMILY_STATUS AS familyStatus,
    NAME_HOUSING_TYPE AS housingType,
    REGION_POPULATION_RELATIVE AS regionPop,
    AGE_YEARS AS ageYears,
    YEARS_EMPLOYED AS yearsEmployed,
    FLAG_UNEMPLOYED AS isUnemployed,
    DAYS_REGISTRATION AS daysReg,
    DAYS_ID_PUBLISH AS daysIdPub,
    OWN_CAR_AGE AS carAge,
    FLAG_MOBIL AS flagMobil,
    FLAG_EMP_PHONE AS empPhone,
    FLAG_WORK_PHONE AS workPhone,
    FLAG_CONT_MOBILE AS contMobile,
    FLAG_PHONE AS flagPhone,
    FLAG_EMAIL AS flagEmail,
    OCCUPATION_TYPE AS occupation,
    CNT_FAM_MEMBERS AS famMembers,
    REGION_RATING_CLIENT AS regionRate,
    REGION_RATING_CLIENT_W_CITY AS regionRateCity,
    WEEKDAY_APPR_PROCESS_START AS weekdayApp,
    HOUR_APPR_PROCESS_START AS hourApp,
    REG_REGION_NOT_LIVE_REGION AS regNotLive,
    REG_REGION_NOT_WORK_REGION AS regNotWork,
    LIVE_REGION_NOT_WORK_REGION AS liveNotWork,
    REG_CITY_NOT_LIVE_CITY AS cityNotLive,
    REG_CITY_NOT_WORK_CITY AS cityNotWork,
    LIVE_CITY_NOT_WORK_CITY AS liveCityNotWork,
    ORGANIZATION_TYPE AS orgType,
    EXT_SOURCE_1 AS ext1,
    EXT_SOURCE_2 AS ext2,
    EXT_SOURCE_3 AS ext3,
    APARTMENTS_AVG AS avgApt,
    BASEMENTAREA_AVG AS avgBsmt,
    YEARS_BEGINEXPLUATATION_AVG AS avgYrsExpl,
    YEARS_BUILD_AVG AS avgYrsBuild,
    COMMONAREA_AVG AS avgCommArea,
    ELEVATORS_AVG AS avgElev,
    ENTRANCES_AVG AS avgEnt,
    FLOORSMAX_AVG AS avgFlrMax,
    FLOORSMIN_AVG AS avgFlrMin,
    LANDAREA_AVG AS avgLand,
    LIVINGAPARTMENTS_AVG AS avgLivApt,
    LIVINGAREA_AVG AS avgLivArea,
    NONLIVINGAPARTMENTS_AVG AS avgNonLivApt,
    NONLIVINGAREA_AVG AS avgNonLivArea,
    APARTMENTS_MODE AS modeApt,
    BASEMENTAREA_MODE AS modeBsmt,
    YEARS_BEGINEXPLUATATION_MODE AS modeYrsExpl,
    YEARS_BUILD_MODE AS modeYrsBuild,
    COMMONAREA_MODE AS modeCommArea,
    ELEVATORS_MODE AS modeElev,
    ENTRANCES_MODE AS modeEnt,
    FLOORSMAX_MODE AS modeFlrMax,
    FLOORSMIN_MODE AS modeFlrMin,
    LANDAREA_MODE AS modeLand,
    LIVINGAPARTMENTS_MODE AS modeLivApt,
    LIVINGAREA_MODE AS modeLivArea,
    NONLIVINGAPARTMENTS_MODE AS modeNonLivApt,
    NONLIVINGAREA_MODE AS modeNonLivArea,
    APARTMENTS_MEDI AS medApt,
    BASEMENTAREA_MEDI AS medBsmt,
    YEARS_BEGINEXPLUATATION_MEDI AS medYrsExpl,
    YEARS_BUILD_MEDI AS medYrsBuild,
    COMMONAREA_MEDI AS medCommArea,
    ELEVATORS_MEDI AS medElev,
    ENTRANCES_MEDI AS medEnt,
    FLOORSMAX_MEDI AS medFlrMax,
    FLOORSMIN_MEDI AS medFlrMin,
    LANDAREA_MEDI AS medLand,
    LIVINGAPARTMENTS_MEDI AS medLivApt,
    LIVINGAREA_MEDI AS medLivArea,
    NONLIVINGAPARTMENTS_MEDI AS medNonLivApt,
    NONLIVINGAREA_MEDI AS medNonLivArea,
    FONDKAPREMONT_MODE AS modeFond,
    HOUSETYPE_MODE AS modeHouse,
    TOTALAREA_MODE AS modeTotal,
    WALLSMATERIAL_MODE AS modeWall,
    EMERGENCYSTATE_MODE AS modeEmerg,
    OBS_30_CNT_SOCIAL_CIRCLE AS obs30,
    DEF_30_CNT_SOCIAL_CIRCLE AS def30,
    OBS_60_CNT_SOCIAL_CIRCLE AS obs60,
    DEF_60_CNT_SOCIAL_CIRCLE AS def60,
    DAYS_LAST_PHONE_CHANGE AS daysPhone,
    FLAG_DOCUMENT_2 AS doc2,
    FLAG_DOCUMENT_3 AS doc3,
    FLAG_DOCUMENT_4 AS doc4,
    FLAG_DOCUMENT_5 AS doc5,
    FLAG_DOCUMENT_6 AS doc6,
    FLAG_DOCUMENT_7 AS doc7,
    FLAG_DOCUMENT_8 AS doc8,
    FLAG_DOCUMENT_9 AS doc9,
    FLAG_DOCUMENT_10 AS doc10,
    FLAG_DOCUMENT_11 AS doc11,
    FLAG_DOCUMENT_12 AS doc12,
    FLAG_DOCUMENT_13 AS doc13,
    FLAG_DOCUMENT_14 AS doc14,
    FLAG_DOCUMENT_15 AS doc15,
    FLAG_DOCUMENT_16 AS doc16,
    FLAG_DOCUMENT_17 AS doc17,
    FLAG_DOCUMENT_18 AS doc18,
    FLAG_DOCUMENT_19 AS doc19,
    FLAG_DOCUMENT_20 AS doc20,
    FLAG_DOCUMENT_21 AS doc21,
    AMT_REQ_CREDIT_BUREAU_HOUR AS reqBurH,
    AMT_REQ_CREDIT_BUREAU_DAY AS reqBurD,
    AMT_REQ_CREDIT_BUREAU_WEEK AS reqBurW,
    AMT_REQ_CREDIT_BUREAU_MON AS reqBurM,
    AMT_REQ_CREDIT_BUREAU_QRT AS reqBurQ,
    AMT_REQ_CREDIT_BUREAU_YEAR AS reqBurY
FROM V_applicationTrainClean
LIMIT 10
""")

applications.createOrReplaceTempView("V_applications")
applications.toPandas()

,loanId,target,contractType,gender,ownCar,ownRealty,childrenCnt,incomeTotal,creditAmt,annuityAmt,...,doc18,doc19,doc20,doc21,reqBurH,reqBurD,reqBurW,reqBurM,reqBurQ,reqBurY
0,100002,1,Cash loans,M,0,1,0,202500.000,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,0,0,0,270000.000,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100007,0,Cash loans,M,0,1,0,121500.000,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100008,0,Cash loans,M,0,1,0,99000.000,490495.5,27517.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,100011,0,Cash loans,F,0,1,0,112500.000,1019610.0,33826.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
5,100015,0,Cash loans,F,0,1,0,38419.155,148365.0,10678.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
6,100016,0,Cash loans,F,0,1,0,67500.000,80865.0,5881.5,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0
7,100019,0,Cash loans,M,1,1,0,157500.000,299772.0,20160.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
8,100030,0,Cash loans,F,0,1,0,90000.000,225000.0,11074.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
9,100031,1,Cash loans,F,0,1,0,112500.000,979992.0,27076.5,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,2.0


In [10]:
display(applicationTrain.limit(5).toPandas())

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
display(applicationTest.limit(5).toPandas())

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
display(bureauRaw.limit(5).toPandas())

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [32]:
bureauRaw.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- CREDIT_CURRENCY: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- CREDIT_TYPE: string (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)



In [14]:
display(bureauBalance.limit(5).toPandas())

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [33]:
bureauBalance.printSchema()

root
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- MONTHS_BALANCE: integer (nullable = true)
 |-- STATUS: string (nullable = true)



In [16]:
display(previousApplication.limit(5).toPandas())

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
display(posCashBalance.limit(5).toPandas())

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [18]:
display(installmentsPayments.limit(5).toPandas())

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [19]:
display(creditCardBalance.limit(5).toPandas())

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [25]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReadData").getOrCreate()

spark.read.parquet("hdfs://namenode:8020/data/silver/staging/POS_CASH_balance_clean").createOrReplaceTempView("V_posCashBalance")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_test_clean").createOrReplaceTempView("V_applicationTestClean")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean").createOrReplaceTempView("V_applicationTrainClean")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_balance_clean").createOrReplaceTempView("V_bureauBalanceClean")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_clean").createOrReplaceTempView("V_bureauClean")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_credit_features").createOrReplaceTempView("V_bureauCreditFeatures")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_delinquency_features").createOrReplaceTempView("V_bureauDelinquencyFeatures")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/credit_card_balance_clean").createOrReplaceTempView("V_creditCardBalanceClean")

In [30]:
table = spark.sql("""
    SELECT *
    FROM tableT
    LIMIT 5
""")
table.createOrReplaceTempView("tableT_baru")
table.toPandas()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1258390,278261,-32,36.0,36.0,Active,0,0
1,1445556,220181,-41,12.0,12.0,Active,0,0
2,2683316,454237,-32,18.0,15.0,Active,0,0
3,2027470,391768,-41,48.0,40.0,Active,0,0
4,2287226,224318,-37,24.0,19.0,Active,0,0


In [36]:
spark.read.parquet("hdfs://namenode:8020/data/silver/home_credit/integrated/train").createOrReplaceTempView("V_trainFull")

In [37]:
train_full = spark.sql("""
SELECT *
FROM V_trainFull
LIMIT 5
""")

train_full.toPandas()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CC_CNT,CC_AVG_BALANCE,CC_TOTAL_LIMIT,CC_AVG_UTILIZATION,CC_MAX_UTILIZATION,CC_AVG_PAYMENT,CC_AVG_ATM_DRAWINGS,CC_OVERLIMIT_RATIO,CC_AVG_DPD,CC_MAX_DPD
0,100002,1,Cash loans,M,0,1,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
1,100003,0,Cash loans,F,0,0,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
2,100007,0,Cash loans,M,0,1,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
3,100008,0,Cash loans,M,0,1,0,99000.0,490495.5,27517.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
4,100011,0,Cash loans,F,0,1,0,112500.0,1019610.0,33826.5,...,1.0,54482.111149,12150000.0,0.302678,1.05,4520.067568,2432.432432,0.0405405405405405,0.0,0.0
